# Weight-Space Lab: Saved Results

This overview reads only the published aggregate JSON and figure. It does not load weights, download data, run inference, or independently re-grade raw responses. For the complete record-based audit, use notebook 07 with the local historical workbench.

GSM8K TRAIN is likely in Qwen's training data; these 30-question, one-model development results are not a capability or benchmark claim.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'repository-layout.json').is_file())
path = ROOT / 'docs' / 'results' / 'p50c_completion.json'
if path.stat().st_size > 2_000_000:
    raise ValueError('Aggregate metadata is unexpectedly large.')
data = json.loads(path.read_text(encoding='utf-8'))
if not isinstance(data, dict) or data.get('status') != 'complete' or data.get('questions') != 30:
    raise ValueError('Expected the completed 30-question aggregate.')
print(data['scope'])

In [ ]:
lines = ['| Outcome at 8,192 tokens | Original | Published p50c |', '|---|---:|---:|']
for outcome, label in [('natural_completion', 'Natural completion (primary)'), ('strict_correct_delivery', 'Strict correct delivery (secondary)'), ('thinking_closed', 'Thinking closed (diagnostic)')]:
    values = []
    for variant in ('original', 'edited'):
        row = data['tables'][outcome][variant][-1]
        if row.get('cap') != 8192 or row.get('n') != 30 or type(row.get('successes')) is not int or not 0 <= row['successes'] <= 30:
            raise ValueError('Invalid aggregate count.')
        values.append(f"{row['successes']}/30 ({row['successes'] / 30:.1%})")
    lines.append(f'| {label} | {values[0]} | {values[1]} |')
display(Markdown('\n'.join(lines)))

## Interpretation And Amendment

The primary natural-completion difference is +26.7 percentage points: 9 edited wins, 1 original win and 20 ties. Exact McNemar p=0.0215; Holm-adjusted p=0.1074 across five caps. **The primary result does not clear that adjustment.** The secondary endpoint is not a replacement primary.

The raw-first text-bound amendment was declared after 11 saved records, before the remaining 19. One regenerated response had 669,118 decoded characters and is strict-incorrect by that rule. The original 11 outcomes are unchanged, and the original failed attempt's 8,192-token reservation remains charged. Exact comparison with its lost raw output is not possible.

See `docs/reports/P50C_COMPLETION_REPORT.md` for Wilson intervals, per-cap paired tests, costs, restoration evidence and limitations.

In [ ]:
display(Image(filename=str(ROOT / 'docs' / 'figures' / 'p50c_vs_original_token_budget.png')))